# QuantLab Research Notebook — reproducible index design

This notebook uses the same registered local FastAPI engine as the React workstation. It runs no broker orders, contains no Dhan secrets and does not make independent historical return claims. Install dependencies and start QuantLab locally before running.

In [ ]:
from quantlab_sdk import QuantLabClient
client = QuantLabClient()
health = client.health()
assert health['broker_orders_enabled'] is False
print('Local engine:', health['ok'])

## 1. Check a dataset passport

Use observed data only with appropriate historical provenance. The synthetic fixture is for validating code paths and UI interactions, not demonstrating real returns.

In [ ]:
datasets = client.datasets()
[(d['id'], d['label'], d['kind'], d['ready']) for d in datasets]

In [ ]:
# Change this to an actual prepared research dataset after checking its evidence.
dataset_id = 'synthetic'
passport = client.data_passport(dataset_id)
print(passport['research_classification'])
print('Unresolved evidence categories:', [item['category'] for item in passport['blockers']])

## 2. Preflight a factor methodology

You can use Momentum, Low Volatility, and, when publication-dated fundamentals exist, Value/Quality/Size. A passing software preflight does NOT certify a benchmark or constituent universe.

In [ ]:
config = {
    'dataset_id': dataset_id,
    'name': 'Notebook demonstration — factor index',
    'factor_weights': {'momentum': 0.5, 'low_volatility': 0.5},
    'top_n': 8, 'max_weight': 0.20, 'weighting': 'equal',
    'rebalance_every': 3, 'cost_bps': 10.0, 'execution_lag': 1,
    'momentum_formation': 12, 'momentum_skip': 1,
    'volatility_lookback': 12, 'sector_cap': None,
    'validation_start': None, 'final_test_start': None,
}
preflight = client.index_preflight(config)
assert preflight['status'] in {'blocked', 'ready_to_execute_unverified'}
print('Blockers:', preflight['blockers'])
print('Warnings:', preflight['warnings'][:5])

## 3. Optional explicit computation

The next cell does not run unless you change ALLOW_RUN to True. It creates a NEW saved research run, never overwrites the canonical price store.

In [ ]:
ALLOW_RUN = False
if ALLOW_RUN:
    assert not preflight['blockers']
    job = client.run_index(config, confirm=True)
    print('Queued job', job['job_id'])
else:
    print('No computation launched. Review the dataset, code, and model assumptions first.')

## 4. Explain an archived result

Select a real archived experiment identifier. Explanation is computed from the saved model and source hashes, not an AI-generated success story.

In [ ]:
runs = client.archived_index_runs()
if runs:
    run_id = runs[0]['id']
    explanation = client.explain('research_run', run_id, level='researcher')
    print(explanation['explanation'])
    print('Evidence class:', explanation['evidence_class'])
else:
    print('No archived index run yet.')

## 5. Factor-risk and publication prerequisites

Active risk requires your own actual factor_returns.csv and dated benchmark_weights.csv. It will fail clearly if either is missing. For publication, use the saved methodology bundle and run the single-command reproducibility checker.

In [ ]:
if runs:
    print(client.methodology(runs[0]['id'])['source_status'])
    print('Reproduce numeric tables with:')
    print('python scripts/reproduce_study.py --run-id', runs[0]['id'])
else:
    print('Create and archive a research run to generate a methodology report.')